# Load Data

In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
def read_parquet_user(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [3]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [4]:
def read_parquet_transaction(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

In [5]:
import os
import polars as pl

def split_and_save_parquet(
    df: pl.DataFrame,
    num_files: int,
    output_dir: str,
    file_prefix: str = "chunk",
    start_index: int = 0
):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.

    Parameters
    ----------
    df : pl.DataFrame
        DataFrame cần tách
    num_files : int
        Số lượng file Parquet
    output_dir : str
        Thư mục lưu file
    file_prefix : str, optional
        Prefix tên file (vd: 'user', 'transaction', 'history'), mặc định 'chunk'
    start_index : int, optional
        Index bắt đầu cho tên file, mặc định 0
    """

    os.makedirs(output_dir, exist_ok=True)

    num_rows = df.height
    rows_per_file = num_rows // num_files

    for i in range(num_files):
        start_row = i * rows_per_file
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows

        split_df = df.slice(start_row, end_row - start_row)

        file_path = os.path.join(
            output_dir,
            f"{file_prefix}_{start_index + i}.parquet"
        )

        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

In [6]:
item = read_parquet_item('preprocessed')
item.head()

item_id,price,category_l1,category_l2,category_l3,category,item_type,gender_target_final,description_final,brand_final,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Dr.Brown's""","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Bộ quần áo""","""Bé Gái""","""Không xác định""","""Con Cưng""","""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Thương hiệu khác""","""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""Không xác định""","""Không xác định""","""﻿﻿Tã dán Merries size S 82 miế…","""Merries""","""[""Từ 4M"", ""3M-6M"", ""12-36M""]"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""Không xác định""","""Không xác định""","""﻿﻿﻿Bỉm tã quần Merries size M …","""Merries""","""12-36M"""


In [7]:
trans = read_parquet_transaction("./preprocessed_01-2025")
trans.head()

item_id,price,quantity,customer_id,created_date,channel,payment,location,discount,category_l1,list_price,category_l2,discount_rate
str,"decimal[38,4]",i32,i32,date,str,str,i32,"decimal[38,4]",str,"decimal[38,4]",str,"decimal[38,4]"
"""7115000000004""",49000.0000,1,5254214,2024-12-24,"""In-Store""","""VietQR""",656,0.0000,"""Thực phẩm cho bé""",49000.0000,"""Snack ăn dặm""",0.0000
"""0029130000030""",69000.0000,1,7573232,2024-12-24,"""In-Store""","""Tiền mặt""",143,0.0000,"""Thực phẩm cho bé""",74000.0000,"""Bột ăn dặm""",0.0676
"""3496000000053""",75000.0000,2,8187418,2024-12-24,"""In-Store""","""MoMo""",213,0.0000,"""Thời trang""",75000.0000,"""Quần áo & Phụ kiện sơ sinh""",0.0000
"""2700000000002""",58500.0000,2,8187418,2024-12-24,"""In-Store""","""MoMo""",213,13000.0000,"""Vệ sinh""",65000.0000,"""Khăn khô""",0.1000
"""0029110000036""",89000.0000,1,6931560,2024-12-28,"""Android""","""MoMo""",590,10000.0000,"""Thực phẩm cho bé""",99000.0000,"""Snack ăn dặm""",0.1010


In [9]:
import polars as pl

trans = ( #lấy 3 tháng gần nhất
    trans
    .filter(
        (pl.col("created_date") >= pl.datetime(2024, 11, 1)) &
        (pl.col("created_date") <  pl.datetime(2025, 2, 1))
    )
)

print("Số dòng sau khi filter:", trans.height)

Số dòng sau khi filter: 9447112


# Discount user: Người dùng đó có mua hàng discount hay k. Tính dựa trên số lần mua trong tháng
Đo xu hướng săn discount dài hạn của user, không phụ thuộc vào một tháng cụ thể.
Trung bình mỗi tháng, user này mua bao nhiêu giao dịch có discount?

In [10]:
user_month = (
    trans
    .with_columns([
        pl.col("created_date").dt.truncate("1mo").alias("month"),
        (pl.col("discount_rate") > 0).cast(pl.Int8).alias("is_discount")
    ])
    .group_by(["customer_id", "month"])
    .agg([
        pl.count().alias("total_tx_count"),
        pl.sum("is_discount").alias("discount_tx_count")
    ])
    .with_columns(
        (pl.col("discount_tx_count") / pl.col("total_tx_count"))
        .fill_null(0)
        .alias("discount_ratio")
    )
)

C:\Users\tncn2\AppData\Local\Temp\ipykernel_17960\1278981641.py:9: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_tx_count"),


In [11]:
user_discount_profile = (
    user_month
    .group_by("customer_id")
    .agg([
        pl.mean("discount_tx_count").alias("avg_discount_tx_per_month"),
        pl.mean("discount_ratio").alias("avg_discount_ratio"),
        pl.count().alias("active_months")
    ])
)

C:\Users\tncn2\AppData\Local\Temp\ipykernel_17960\1261084741.py:7: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("active_months")


In [12]:
X = user_discount_profile.select([
    "avg_discount_tx_per_month",
    "avg_discount_ratio"
]).to_numpy()


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [14]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

cluster_raw = kmeans.fit_predict(X_scaled)

In [15]:
user_discount_profile = user_discount_profile.with_columns(
    pl.Series("discount_cluster_raw", cluster_raw)
)

In [16]:
cluster_order = (
    user_discount_profile
    .group_by("discount_cluster_raw")
    .agg([
        pl.mean("avg_discount_tx_per_month").alias("mean_tx"),
        pl.mean("avg_discount_ratio").alias("mean_ratio")
    ])
    .sort(["mean_tx", "mean_ratio"])
    .with_row_index("discount_behavior_level")  
)

In [17]:
user_discount_profile = (
    user_discount_profile
    .join(
        cluster_order.select([
            "discount_cluster_raw",
            "discount_behavior_level"
        ]),
        on="discount_cluster_raw",
        how="left"
    )
    .drop("discount_cluster_raw")
)

In [18]:
user_discount_profile.head()

customer_id,avg_discount_tx_per_month,avg_discount_ratio,active_months,discount_behavior_level
i32,f64,f64,u32,u32
4538120,2.0,1.0,1,1
5861786,1.0,1.0,1,1
2989806,1.0,1.0,1,1
8001529,2.0,0.833333,3,1
6467506,0.5,0.5,2,0


- avg_discount_tx_per_month: Trung bình, mỗi tháng user mua bao nhiêu giao dịch giảm giá?
- avg_discount_ratio: tỷ lệ trung bình các giao dịch có giảm giá của một người dùng, được tính trên tất cả các tháng người dùng có phát sinh giao dịch
- active_months: Số tháng user có ít nhất 1 giao dịch
- discount_behavior_level: mức độ mua hàng discount của user (0,1,2)


In [19]:
user_discount_profile.group_by("discount_behavior_level").agg(
    pl.count().alias("num_users")
).sort("discount_behavior_level")

C:\Users\tncn2\AppData\Local\Temp\ipykernel_17960\632212740.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_users")


discount_behavior_level,num_users
u32,u32
0,472792
1,582453
2,114374


In [20]:
user_discount_profile.group_by("discount_behavior_level").agg([
    pl.mean("avg_discount_tx_per_month").alias("mean_discount_tx_per_month"),
    pl.median("avg_discount_tx_per_month").alias("median_discount_tx_per_month"),
    pl.mean("avg_discount_ratio").alias("mean_discount_ratio"),
    pl.mean("active_months").alias("mean_active_months")
]).sort("discount_behavior_level")

discount_behavior_level,mean_discount_tx_per_month,median_discount_tx_per_month,mean_discount_ratio,mean_active_months
u32,f64,f64,f64,f64
0,0.975098,1.0,0.235319,1.528088
1,2.233467,2.0,0.880242,1.608696
2,9.140361,8.0,0.682939,2.354923


In [21]:
user_discount_profile.head()

customer_id,avg_discount_tx_per_month,avg_discount_ratio,active_months,discount_behavior_level
i32,f64,f64,u32,u32
4538120,2.0,1.0,1,1
5861786,1.0,1.0,1,1
2989806,1.0,1.0,1,1
8001529,2.0,0.833333,3,1
6467506,0.5,0.5,2,0


In [23]:
split_and_save_parquet(
    df=user_discount_profile,
    num_files=1,
    output_dir="feature_engineered"
)

Đã lưu file: feature_engineered\sale_pers.user_chunk_0.parquet


In [27]:
split_and_save_parquet(
    df=user_discount_profile,
    num_files=1,
    output_dir="feature_engineered",
    file_prefix="discount_pers_user"
)

Đã lưu file: feature_engineered\discount_pers_user_0.parquet


# User Category

In [48]:
# Chỉ giữ cột cần thiết để tránh nặng
item_cat = item.select(["item_id", "category_l1"])

# Join category vào transaction
trans_cat = (
    trans
    .select(["customer_id", "item_id"])
    .join(item_cat, on="item_id", how="left")
    .filter(pl.col("category_l1").is_not_null())
)

In [30]:
user_cat_cnt = (
    trans_cat
    .group_by(["customer_id", "category_l1"])
    .agg(
        pl.count().alias("cnt")
    )
)

C:\Users\tncn2\AppData\Local\Temp\ipykernel_17960\3875378938.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("cnt")


In [31]:
user_cat_wide = (
    user_cat_cnt
    .pivot(
        index="customer_id",
        columns="category_l1",
        values="cnt"
    )
    .fill_null(0)
)

C:\Users\tncn2\AppData\Local\Temp\ipykernel_17960\4116182049.py:3: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(


In [32]:
user_cat_wide = (
    user_cat_cnt
    .pivot(
        index="customer_id",
        columns="category_l1",
        values="cnt"
    )
    .fill_null(0)
)

C:\Users\tncn2\AppData\Local\Temp\ipykernel_17960\4116182049.py:3: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(


In [33]:
user_cat_wide.head()

customer_id,Sữa nước,Tã,Phụ kiện,Thực phẩm cho bé,Sữa,Hóa mỹ phẩm cho bé,Đồ chơi & Sách,Thời trang,TPCN,Babycare,Thực phẩm cho gia đình,Textile,Vệ sinh,Hóa mỹ phẩm gia đình,Gói Hội Viên
i32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
8190218,2,0,0,3,0,0,0,0,0,0,1,0,0,0,0
3014271,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0
5460513,0,0,1,9,0,0,1,0,1,0,0,0,1,0,0
7989901,2,3,0,1,0,0,0,0,0,0,0,0,0,0,0
501189,4,0,2,1,6,0,5,1,0,0,2,1,0,0,0


In [34]:
rename_map = {
    col: f"cnt_cat_{col}"
    for col in user_cat_wide.columns
    if col != "customer_id"
}

user_cat_wide = user_cat_wide.rename(rename_map)

In [35]:
user_cat_wide = user_cat_wide.with_columns(
    pl.sum_horizontal(
        [pl.col(c) for c in user_cat_wide.columns if c != "customer_id"]
    ).alias("total_cat_tx")
)

In [36]:
ratio_exprs = []

for col in user_cat_wide.columns:
    if col.startswith("cnt_cat_"):
        ratio_exprs.append(
            (pl.col(col) / pl.col("total_cat_tx"))
            .alias(col.replace("cnt_", "ratio_"))
        )

user_cat_wide = user_cat_wide.with_columns(ratio_exprs)

In [37]:
user_cat_ratio = user_cat_wide.drop("total_cat_tx")

In [38]:
user_cat_ratio.head()

customer_id,cnt_cat_Sữa nước,cnt_cat_Tã,cnt_cat_Phụ kiện,cnt_cat_Thực phẩm cho bé,cnt_cat_Sữa,cnt_cat_Hóa mỹ phẩm cho bé,cnt_cat_Đồ chơi & Sách,cnt_cat_Thời trang,cnt_cat_TPCN,cnt_cat_Babycare,cnt_cat_Thực phẩm cho gia đình,cnt_cat_Textile,cnt_cat_Vệ sinh,cnt_cat_Hóa mỹ phẩm gia đình,cnt_cat_Gói Hội Viên,ratio_cat_Sữa nước,ratio_cat_Tã,ratio_cat_Phụ kiện,ratio_cat_Thực phẩm cho bé,ratio_cat_Sữa,ratio_cat_Hóa mỹ phẩm cho bé,ratio_cat_Đồ chơi & Sách,ratio_cat_Thời trang,ratio_cat_TPCN,ratio_cat_Babycare,ratio_cat_Thực phẩm cho gia đình,ratio_cat_Textile,ratio_cat_Vệ sinh,ratio_cat_Hóa mỹ phẩm gia đình,ratio_cat_Gói Hội Viên
i32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
8190218,2,0,0,3,0,0,0,0,0,0,1,0,0,0,0,0.333333,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.0,0.0,0.0,0.0
3014271,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0.0,0.333333,0.0,0.0,0.0,0.0,0.333333,0.0,0.0,0.0,0.0,0.0,0.333333,0.0,0.0
5460513,0,0,1,9,0,0,1,0,1,0,0,0,1,0,0,0.0,0.0,0.076923,0.692308,0.0,0.0,0.076923,0.0,0.076923,0.0,0.0,0.0,0.076923,0.0,0.0
7989901,2,3,0,1,0,0,0,0,0,0,0,0,0,0,0,0.333333,0.5,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
501189,4,0,2,1,6,0,5,1,0,0,2,1,0,0,0,0.181818,0.0,0.090909,0.045455,0.272727,0.0,0.227273,0.045455,0.0,0.0,0.090909,0.045455,0.0,0.0,0.0


In [39]:
cols_to_drop = [c for c in user_cat_ratio.columns if c.startswith("cnt_")]

user_cat_ratio = user_cat_ratio.drop(cols_to_drop)


In [40]:
user_cat_ratio.head()

customer_id,ratio_cat_Sữa nước,ratio_cat_Tã,ratio_cat_Phụ kiện,ratio_cat_Thực phẩm cho bé,ratio_cat_Sữa,ratio_cat_Hóa mỹ phẩm cho bé,ratio_cat_Đồ chơi & Sách,ratio_cat_Thời trang,ratio_cat_TPCN,ratio_cat_Babycare,ratio_cat_Thực phẩm cho gia đình,ratio_cat_Textile,ratio_cat_Vệ sinh,ratio_cat_Hóa mỹ phẩm gia đình,ratio_cat_Gói Hội Viên
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
8190218,0.333333,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.0,0.0,0.0,0.0
3014271,0.0,0.333333,0.0,0.0,0.0,0.0,0.333333,0.0,0.0,0.0,0.0,0.0,0.333333,0.0,0.0
5460513,0.0,0.0,0.076923,0.692308,0.0,0.0,0.076923,0.0,0.076923,0.0,0.0,0.0,0.076923,0.0,0.0
7989901,0.333333,0.5,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
501189,0.181818,0.0,0.090909,0.045455,0.272727,0.0,0.227273,0.045455,0.0,0.0,0.090909,0.045455,0.0,0.0,0.0


In [42]:
ratio_cols = [c for c in user_cat_ratio.columns if c.startswith("ratio_cat_")]

user_cat_ratio.with_columns(
    (pl.max_horizontal([pl.col(c) for c in ratio_cols]) >= 0.9)
    .alias("is_single_cat")
).group_by("is_single_cat").agg(
    pl.count().alias("num_users")
)

C:\Users\tncn2\AppData\Local\Temp\ipykernel_17960\701240895.py:7: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_users")


is_single_cat,num_users
bool,u32
false,731726
true,437893


In [45]:
user_cat_ratio.with_columns(
    pl.max_horizontal([pl.col(c) for c in ratio_cols]).alias("top_cat_ratio")
).select("top_cat_ratio").describe()

statistic,top_cat_ratio
str,f64
"""count""",1.169619e6
"""null_count""",0.0
"""mean""",0.688831
"""std""",0.271211
"""min""",0.1
"""25%""",0.494624
"""50%""",0.666667
"""75%""",1.0
"""max""",1.0


In [ ]:
output_path = "feature_engineered/user_cat_ratio.parquet"

user_cat_ratio.write_parquet(output_path)

print(f"Đã lưu user_cat_ratio tại: {output_path}")

Đã lưu user_cat_ratio tại: feature_engineered/user_cat_ratio.parquet


# age

In [8]:
import re


def get_default_max_by_min(age_min: int) -> int:
    if age_min < 60:
        return 60
    elif age_min < 120:
        return 120
    elif age_min < 144:
        return 180
    else:
        return 180




PATTERNS = {
    "LIST": re.compile(r"^\s*\[.*\]\s*$"),
    "MIXED_RANGE": re.compile(r"(\d+)\s*[mM]\s*-\s*(\d+)\s*[yY]", re.IGNORECASE),
    "RANGE": re.compile(r"(\d+)\s*-\s*(\d+)\s*[mM]", re.IGNORECASE),
    "FROM": re.compile(r"(?:từ|>=)\s*(\d+)\s*[mM]", re.IGNORECASE),
    "SINGLE": re.compile(r"\b(\d+)\s*[mM]\b", re.IGNORECASE),
}




def parse_age_group_value(val: str):
    null_ret = {"age_min_month": None, "age_max_month": None}


    if val is None:
        return null_ret


    s = val.strip()
    s_norm = s.lower()


    if s_norm in {"mẹ", "không xác định"}:
        return null_ret


    if re.search(r"\b\d+\s*d\+\b", s_norm):
        return null_ret


    # LIST
    if PATTERNS["LIST"].match(s):
        mins, maxs = [], []


        for m in PATTERNS["RANGE"].finditer(s):
            mins.append(int(m.group(1)))
            maxs.append(int(m.group(2)))


        for m in PATTERNS["MIXED_RANGE"].finditer(s):
            mins.append(int(m.group(1)))
            maxs.append(int(m.group(2)) * 12)


        for m in PATTERNS["FROM"].finditer(s):
            mins.append(int(m.group(1)))


        for m in PATTERNS["SINGLE"].finditer(s):
            mins.append(int(m.group(1)))


        if not mins:
            return null_ret


        age_min = min(mins)
        age_max = max(maxs) if maxs else get_default_max_by_min(age_min)


        return {
            "age_min_month": age_min,
            "age_max_month": age_max,
        }


    # MIXED
    m = PATTERNS["MIXED_RANGE"].search(s)
    if m:
        return {
            "age_min_month": int(m.group(1)),
            "age_max_month": int(m.group(2)) * 12,
        }


    # RANGE
    m = PATTERNS["RANGE"].search(s)
    if m:
        return {
            "age_min_month": int(m.group(1)),
            "age_max_month": int(m.group(2)),
        }


    # FROM
    m = PATTERNS["FROM"].search(s)
    if m:
        age_min = int(m.group(1))
        return {
            "age_min_month": age_min,
            "age_max_month": get_default_max_by_min(age_min),
        }


    # SINGLE
    m = PATTERNS["SINGLE"].search(s)
    if m:
        age_min = int(m.group(1))
        return {
            "age_min_month": age_min,
            "age_max_month": get_default_max_by_min(age_min),
        }


    return null_ret


In [12]:
def parse_age_group_safe(val):
    try:
        res = parse_age_group_value(val)
        if res is None:
            return {"age_min_month": None, "age_max_month": None}
        return res
    except Exception:
        return {"age_min_month": None, "age_max_month": None}

In [13]:
import polars as pl

def build_item_age_features(
    item_df: pl.DataFrame,
    col: str = "age_group_final",
) -> pl.DataFrame:

    return (
        item_df
        # --- parse age_group_final ---
        .with_columns(
            pl.col(col)
            .cast(pl.Utf8)
            .map_elements(
                lambda x: parse_age_group_safe(x),
                return_dtype=pl.Struct([
                    pl.Field("age_min_month", pl.Int64),
                    pl.Field("age_max_month", pl.Int64),
                ]),
                skip_nulls=False,
            )
            .alias("age_struct")
        )

        # --- tách struct ---
        .with_columns([
            pl.col("age_struct").struct.field("age_min_month"),
            pl.col("age_struct").struct.field("age_max_month"),
        ])

        # --- có age hay không ---
        .with_columns(
            pl.col("age_min_month")
            .is_not_null()
            .cast(pl.Int8)
            .alias("has_age_info")
        )

        # --- age_range ---
        .with_columns(
            pl.when(pl.col("has_age_info") == 1)
              .then(pl.col("age_max_month") - pl.col("age_min_month"))
              .otherwise(None)
              .alias("age_range")
        )

        # --- log(age_range) ---
        .with_columns(
            pl.when(pl.col("age_range").is_not_null())
              .then((pl.col("age_range") + 1).log())
              .otherwise(None)
              .alias("age_range_log")
        )

        # --- cleanup ---
        .drop("age_struct")
    )


In [14]:
tests = [
    "Từ 9M",
    "Từ 36M",
    "Từ 72M",
    "Từ 144M",
    "0-12M",
    "6-18M",
    "6M-5Y",
    '["Từ 4M","3M-6M","12-36M"]',
    "Mẹ",
    "Không xác định",
]

for t in tests:
    print(t, "→", parse_age_group_value(t))

Từ 9M → {'age_min_month': 9, 'age_max_month': 60}
Từ 36M → {'age_min_month': 36, 'age_max_month': 60}
Từ 72M → {'age_min_month': 72, 'age_max_month': 120}
Từ 144M → {'age_min_month': 144, 'age_max_month': 180}
0-12M → {'age_min_month': 0, 'age_max_month': 12}
6-18M → {'age_min_month': 6, 'age_max_month': 18}
6M-5Y → {'age_min_month': 6, 'age_max_month': 60}
["Từ 4M","3M-6M","12-36M"] → {'age_min_month': 3, 'age_max_month': 36}
Mẹ → {'age_min_month': None, 'age_max_month': None}
Không xác định → {'age_min_month': None, 'age_max_month': None}


In [15]:
item_feat = build_item_age_features(
    item,
    col="age_group_final"
)

In [21]:
import os

output_path = "item_age.parquet"

item_feat.write_parquet(output_path)

print("Saved to:", output_path)

Saved to: item_age.parquet


In [23]:
import polars as pl

item_feat = item_feat.with_columns(
    pl.when(pl.col("has_age_info") == 1)
      .then(pl.col("age_max_month") - pl.col("age_min_month"))
      .otherwise(None)
      .alias("age_range")
)

In [25]:
item_feat.select([
    pl.count().alias("total_items"),
    (pl.col("age_range") < 0).sum().alias("neg_range_cnt"),
])

C:\Users\tncn2\AppData\Local\Temp\ipykernel_6020\1789573508.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_items"),


total_items,neg_range_cnt
u32,u32
27323,20


In [26]:
neg_df = item_feat.filter(pl.col("age_range") < 0)

neg_df.select([
    "item_id",
    "age_group_final",
    "age_min_month",
    "age_max_month",
    "age_range",
]).head(50)

item_id,age_group_final,age_min_month,age_max_month,age_range
str,str,i64,i64,i64
"""2890000000001""","""Từ 216M""",216,180,-36
"""2890000000003""","""Từ 216M""",216,180,-36
"""4461000000003""","""Từ 216M""",216,180,-36
"""2890000000002""","""Từ 216M""",216,180,-36
"""4459000000001""","""Từ 216M""",216,180,-36
…,…,…,…,…
"""6513000000001""","""Từ 228M""",228,180,-48
"""6512000000002""","""Từ 228M""",228,180,-48
"""2890000000004""","""Từ 216M""",216,180,-36


In [27]:
(
    neg_df
    .group_by("age_group_final")
    .agg(pl.count().alias("cnt"))
    .sort("cnt", descending=True)
)

C:\Users\tncn2\AppData\Local\Temp\ipykernel_6020\107619283.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("cnt"))


age_group_final,cnt
str,u32
"""Từ 216M""",14
"""Từ 228M""",4
"""Từ 192M""",1
"""Từ 480M""",1
